# 🤖 AI Code Review & Refactoring Agent

A practical Generative AI project that analyzes source code, detects bugs and code smells, suggests improvements, generates safer refactoring ideas, and validates the result.

**Workflow:** Code → Analyzer → Review Agent → Refactoring Agent → Validation Agent → Final Report


In [ ]:
%pip install -q openai python-dotenv pydantic

In [ ]:
import os
import re
import json
from typing import List, Optional
from pydantic import BaseModel

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = os.getenv("OPENROUTER_MODEL", "openai/gpt-4.1-mini")

print("OpenRouter configured:", bool(OPENROUTER_API_KEY))
print("Model:", OPENROUTER_MODEL)


## 🧩 1. Structured Review Models

In [ ]:
class CodeIssue(BaseModel):
    severity: str
    category: str
    line: Optional[int] = None
    title: str
    explanation: str
    recommendation: str

class ReviewSummary(BaseModel):
    risk_level: str
    issue_count: int
    severity_counts: dict

print("Review schemas ready.")


## 🔍 2. Deterministic Code Analyzer

Fast local checks can identify common security, correctness, reliability, and maintainability problems before invoking an LLM.


In [ ]:
def analyze_python(code: str) -> List[CodeIssue]:
    issues = []

    for line_no, line in enumerate(code.splitlines(), 1):
        s = line.strip()

        if "eval(" in s:
            issues.append(CodeIssue(
                severity="HIGH",
                category="Security",
                line=line_no,
                title="Use of eval()",
                explanation="eval() can execute arbitrary expressions.",
                recommendation="Use explicit parsing or a safe mapping."
            ))

        if s.startswith("except:"):
            issues.append(CodeIssue(
                severity="MEDIUM",
                category="Reliability",
                line=line_no,
                title="Bare exception handler",
                explanation="A bare except can hide unexpected failures.",
                recommendation="Catch specific exception types."
            ))

        if "print(" in s:
            issues.append(CodeIssue(
                severity="LOW",
                category="Maintainability",
                line=line_no,
                title="Direct print statement",
                explanation="Production applications normally benefit from structured logging.",
                recommendation="Use the logging module."
            ))

        if re.search(r"password\s*=\s*['\"]", s, re.I):
            issues.append(CodeIssue(
                severity="CRITICAL",
                category="Security",
                line=line_no,
                title="Possible hard-coded secret",
                explanation="A password-like value is directly embedded in source.",
                recommendation="Use environment variables or a secret manager."
            ))

    if re.search(r"def\s+\w+\([^)]*=\s*\[", code):
        issues.append(CodeIssue(
            severity="HIGH",
            category="Correctness",
            title="Mutable default argument",
            explanation="A mutable default value can be shared between function calls.",
            recommendation="Use None and initialize the list inside the function."
        ))

    return issues


## 🧪 3. Example Code Review

In [ ]:
sample_code = """
def add_user(name, users=[]):
    users.append(name)
    return users

def run_expression(value):
    try:
        return eval(value)
    except:
        return None

password = "demo-secret"
print(run_expression("2 + 3"))
"""

issues = analyze_python(sample_code)

for issue in issues:
    where = f"line {issue.line}" if issue.line else "general"
    print(f"[{issue.severity}] {issue.category} | {where} | {issue.title}")
    print("  ->", issue.recommendation)


## 🧠 4. AI Code Review Agent

When an OpenRouter API key is configured, the LLM can perform deeper analysis covering architecture, performance, testing, security, duplication, naming, and maintainability.


In [ ]:
def llm_review(code: str, language: str = "python") -> str:
    if not OPENROUTER_API_KEY:
        return "No API key configured. Local deterministic review is still available."

    from openai import OpenAI

    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY
    )

    system_prompt = (
        "You are a senior software engineer. Review code rigorously. "
        "Only report issues supported by the supplied code. Cover correctness, "
        "security, performance, maintainability, and testing."
    )

    user_prompt = f"""Review this {language} code.

Return:
1. Executive summary
2. Critical/high/medium/low issues
3. Refactoring recommendations
4. Testing recommendations
5. Security concerns

CODE:
```{language}
{code}
```"""

    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

print(llm_review(sample_code))


## ♻️ 5. Refactoring Agent

The prototype demonstrates targeted transformations. In production, an LLM can generate a patch, but the patch must be tested and validated before acceptance.


In [ ]:
def demo_refactor(code: str) -> str:
    result = code

    result = result.replace(
        "def add_user(name, users=[]):",
        "def add_user(name, users=None):"
    )

    result = result.replace(
        "    users.append(name)\n    return users",
        "    if users is None:\n        users = []\n    users.append(name)\n    return users"
    )

    result = result.replace(
        'password = "demo-secret"',
        'password = os.getenv("APP_PASSWORD")'
    )

    return result

refactored_code = demo_refactor(sample_code)
print(refactored_code)


## ✅ 6. Validation Agent

In [ ]:
def validate_python(code: str) -> dict:
    try:
        compile(code, "<reviewed_code>", "exec")
        return {"valid": True, "error": None}
    except SyntaxError as exc:
        return {
            "valid": False,
            "error": f"SyntaxError on line {exc.lineno}: {exc.msg}"
        }

validation = validate_python(refactored_code)
print(json.dumps(validation, indent=2))


## 📊 7. Review Summary

In [ ]:
def summarize(issues: List[CodeIssue]) -> ReviewSummary:
    counts = {}

    for issue in issues:
        counts[issue.severity] = counts.get(issue.severity, 0) + 1

    if counts.get("CRITICAL"):
        risk = "CRITICAL"
    elif counts.get("HIGH"):
        risk = "HIGH"
    elif counts.get("MEDIUM"):
        risk = "MEDIUM"
    else:
        risk = "LOW"

    return ReviewSummary(
        risk_level=risk,
        issue_count=len(issues),
        severity_counts=counts
    )

summary = summarize(issues)
print(summary.model_dump())


## 🚀 8. End-to-End Pipeline

**Analyze → Review → Refactor → Validate**

This can become the foundation for a GitHub Pull Request bot or CI/CD quality gate.


In [ ]:
def code_review_pipeline(code: str, language: str = "python") -> dict:
    issues = analyze_python(code) if language.lower() == "python" else []
    summary = summarize(issues)

    result = {
        "summary": summary.model_dump(),
        "issues": [issue.model_dump() for issue in issues],
        "refactored_code": None,
        "validation": None
    }

    if language.lower() == "python":
        result["refactored_code"] = demo_refactor(code)
        result["validation"] = validate_python(result["refactored_code"])

    if OPENROUTER_API_KEY:
        result["llm_review"] = llm_review(code, language)

    return result

result = code_review_pipeline(sample_code)

print(json.dumps(
    {
        "summary": result["summary"],
        "validation": result["validation"]
    },
    indent=2
))


## 🏗️ 9. Production Architecture

```text
GitHub Pull Request
        ↓
   Code Collector
        ↓
    Planner Agent
        ↓
 ┌──────┼──────────┐
 ↓      ↓          ↓
Security Quality Performance
Agent    Agent     Agent
 └──────┼──────────┘
        ↓
   Review Merger
        ↓
 Refactoring Agent
        ↓
 Validation Agent
        ↓
Tests / Lint / Security Scan
        ↓
 Pull Request Comment
        ↓
 Human Approval
```

Recommended integrations: **GitHub, GitHub Actions, LangGraph, MCP, OpenRouter, AST, Ruff, MyPy, Pytest, Semgrep, SonarQube.**


## 🧠 10. Advanced Features

- 🌳 Repository-level code understanding
- 🧩 Python AST analysis
- 🧪 Automatic unit-test generation
- 🔐 Security vulnerability detection
- ⚡ Performance analysis
- 🔄 Self-repair loop
- 🔌 MCP-based developer tools
- 🔀 LangGraph multi-agent orchestration
- 📚 Repository RAG
- 🤖 Automated GitHub PR review

### Self-Repair Loop

```text
Review → Refactor → Test → Failed?
                         ↓
                    Repair Code
                         ↓
                      Re-test
                         ↓
                        Pass
```


## 📈 11. Evaluation Metrics

| Metric | Purpose |
|---|---|
| 🎯 Issue Precision | Percentage of reported issues that are real |
| 🔎 Issue Recall | Important issues successfully detected |
| ♻️ Refactor Correctness | Whether intended behavior is preserved |
| 🧪 Test Pass Rate | Generated code passing tests |
| 🛡️ Security Detection | Vulnerabilities correctly identified |
| 💰 Token Cost | LLM cost per review |
| ⏱️ Latency | Time required for review |
| 👨‍💻 Developer Acceptance | Suggestions accepted by developers |

## 🛡️ Production Guardrails

- Never execute untrusted generated code directly.
- Use an isolated sandbox/container for generated code.
- Limit filesystem and network permissions.
- Keep secrets outside the model context where possible.
- Require tests before applying automated refactors.
- Use human approval for high-risk changes.
- Keep bounded retry loops and audit logs.

## 🚀 Roadmap

**Phase 1:** Code analyzer → AI review → refactoring → validation

**Phase 2:** LangGraph → MCP → test generation → GitHub integration → self-repair

**Phase 3:** Repository RAG → security scanning → evaluation dashboard → production CI/CD

## 🎯 Learning Outcomes

`LLMs` • `AI Agents` • `Code Intelligence` • `LangGraph` • `MCP` • `RAG` • `GitHub Automation` • `CI/CD` • `AI Evaluation` • `Guardrails`

**Vision:** Build an AI software-engineering teammate that can review, explain, improve, test, and safely propose changes to real-world codebases. 🚀
